# Session 2 — Implementing Data Versioning Using DVC

**Goal:** track a dataset's history the way Git tracks code — commit-able snapshots,
diffable metadata, and the ability to check out any previous version — using
[DVC](https://dvc.org) (Data Version Control).

## Why not just commit the CSV to Git?

Git is built for text diffs of source code, not multi-megabyte (or gigabyte) binary
data files — every version bloats the repo forever. DVC's trick: it stores the actual
data in a separate location (a "remote" — local folder, S3, GCS, etc.) and commits a
tiny `.dvc` **pointer file** (just a hash and size) to Git instead. Git tracks which
version of the pointer you're on; DVC uses the pointer to fetch the matching data.

## Prerequisites

```bash
pip install dvc
```
This notebook uses a **local folder** as the DVC remote, so everything runs without
any cloud account — swap `dvc remote add` for an S3/GCS URL in a real team setup.

In [ ]:
import subprocess, os, shutil

WORKDIR = "dvc_demo"
shutil.rmtree(WORKDIR, ignore_errors=True)
os.makedirs(WORKDIR, exist_ok=True)

def run(cmd, cwd=WORKDIR):
    result = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

## Step 1 — Initialize a Git repo and a DVC project

DVC sits on top of Git — it needs an initialized Git repo to hook into.

In [ ]:
run("git init -q")
run("dvc init -q")
run("git status")

## Step 2 — Create a dataset and add it to DVC

`dvc add` moves the real file into DVC's internal cache and replaces it in your
working directory with the same filename, while writing a small `.dvc` pointer file
next to it (that pointer is what actually goes into Git).

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
df_v1 = pd.DataFrame({
    "feature_a": rng.normal(size=200),
    "feature_b": rng.normal(size=200),
    "label": rng.integers(0, 2, size=200),
})
df_v1.to_csv(f"{WORKDIR}/dataset.csv", index=False)
print(df_v1.shape)

In [ ]:
run("dvc add dataset.csv")
run("cat dataset.csv.dvc")

## Step 3 — Commit the pointer file to Git (not the data)

Notice `dataset.csv` itself is now gitignored (DVC wrote that automatically) — only
the tiny `.dvc` pointer and the `.gitignore` entry go into Git history.

In [ ]:
run("cat .gitignore")
run('git add dataset.csv.dvc .gitignore dvc.yaml 2>/dev/null; git add dataset.csv.dvc .gitignore')
run('git -c user.email=demo@example.com -c user.name=demo commit -q -m "v1: initial dataset"')
run("git log --oneline")

## Step 4 — Configure a remote and push the data

The pointer lives in Git; the actual bytes live in the DVC **remote**. Here the
remote is just another local folder, standing in for what would be an S3 bucket or
GCS bucket in a real deployment.

In [ ]:
os.makedirs(f"{WORKDIR}/../dvc_remote_storage", exist_ok=True)
run("dvc remote add -d localremote ../dvc_remote_storage")
run('git add .dvc/config')
run('git -c user.email=demo@example.com -c user.name=demo commit -q -m "configure dvc remote"')
run("dvc push")

## Step 5 — Change the data, and version the new snapshot

This is the payoff: modify the dataset, `dvc add` again, commit the updated pointer.
Git history now records *when* the data changed, even though the data itself never
touched Git directly.

In [ ]:
df_v2 = pd.concat([df_v1, pd.DataFrame({
    "feature_a": rng.normal(size=50),
    "feature_b": rng.normal(size=50),
    "label": rng.integers(0, 2, size=50),
})], ignore_index=True)
df_v2.to_csv(f"{WORKDIR}/dataset.csv", index=False)
print(f"v1 had {len(df_v1)} rows, v2 has {len(df_v2)} rows")

run("dvc add dataset.csv")
run('git add dataset.csv.dvc')
run('git -c user.email=demo@example.com -c user.name=demo commit -q -m "v2: add 50 more rows"')
run("dvc push")
run("git log --oneline")

## Step 6 — Roll back to the previous data version

Check out an older Git commit, then `dvc checkout` to sync the working file with
whatever pointer that commit recorded — the dataset itself reverts to its earlier
shape.

In [ ]:
log = run("git log --oneline").stdout.strip().split("\n")
v1_commit = log[-1].split()[0]  # oldest commit = "v1: initial dataset"
print("Rolling back to:", v1_commit)

run(f"git checkout -q {v1_commit} -- dataset.csv.dvc")
run("dvc checkout")

reverted = pd.read_csv(f"{WORKDIR}/dataset.csv")
print(f"Rows after rollback: {len(reverted)} (expected {len(df_v1)})")

## Step 7 — A minimal DVC pipeline (`dvc.yaml`)

Beyond versioning a single file, DVC can track a whole **pipeline**: stages with
dependencies and outputs, re-run only when an input actually changed. This mirrors
what `dvc.yaml` would look like for a train stage feeding off `dataset.csv`.

In [ ]:
pipeline_yaml = '''\
stages:
  train:
    cmd: python train.py
    deps:
      - dataset.csv
      - train.py
    outs:
      - model.pkl
    metrics:
      - metrics.json:
          cache: false
'''
with open(f"{WORKDIR}/dvc.yaml", "w") as f:
    f.write(pipeline_yaml)

print(pipeline_yaml)
print("In a real project: `dvc repro` re-runs `train.py` only if dataset.csv, ")
print("train.py, or their recorded hashes have changed since the last run.")

## What to try next

* Point `dvc remote add` at a real S3/GCS bucket instead of a local folder — the rest
  of the workflow (`dvc add`, `dvc push`, `dvc pull`, `dvc checkout`) is identical.
* Write an actual `train.py` and run `dvc repro` to see dependency-aware re-execution.
* Session 3 (DagsHub) shows how to host both the Git history *and* the DVC remote in
  one place, so a team can `git clone` + `dvc pull` from a single URL.